# CorroborIA – exploration, approche et résultats
Défi Loto-Québec : distinguer **écarts justifiés** et **vraies anomalies** entre le Système A – RH et le Système B – Temps.

Ce notebook suit les trois niveaux recommandés par les consignes :
1. comparaison brute des valeurs (et normalisation) ;
2. application des règles métier du mapping ;
3. analyse des cas qui demeurent ambigus (IA).

In [1]:
import os, sys
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
import pandas as pd
pd.set_option("display.max_colwidth", 90); pd.set_option("display.width", 200)
from datetime import date
from corroborai.chargement import charger
from corroborai.config import Parametres, CHAMPS
from corroborai.moteur import corroborer
d = charger(dossier="data")
print("Source :", d.source.shape, "| Cible :", d.cible.shape, "| Détail du poste :", d.detail_poste.shape,
      "| Motifs :", d.motifs.shape)

Source : (23, 34) | Cible : (22, 80) | Détail du poste : (114, 12) | Motifs : (92, 3)


## 1. Exploration – ce que les données révèlent
* La cible compte **79 colonnes**, dont une majorité vides : seuls les champs du **mapping** sont corroborés.
* Un employé peut avoir **plusieurs affectations** (P/A/S) → la clé n'est pas le matricule seul.
* Formats hétérogènes : dates `1995-02-09 00:00:00` vs `1995-02-09T00:00:00.000Z`, booléens `Oui/Non` vs `true/false`, encodage `Absence complÃ¨te`.
* L'extraction *détail du poste* est un **CSV stocké dans une seule cellule Excel**, avec des dates en **numéro de série Excel**.

In [2]:
vides = [c for c in d.cible.columns if d.cible[c].isna().all()]
print(f"{len(vides)} colonnes cible entièrement vides")
print("Matricules multi-affectations :", d.source["Matricule"].value_counts().loc[lambda s: s > 1].to_dict())
print("Statuts cible :", d.cible["detailedStatus"].unique())
d.detail_poste.head(3)

49 colonnes cible entièrement vides
Matricules multi-affectations : {'7683990': 3, '1545850': 2}
Statuts cible : <ArrowStringArray>
['Actif', 'Absence complÃ¨te']
Length: 2, dtype: str


,IdentifiantPoste,IdentifiantEmploi,CodeDirectionAffectée,DateEffetAffectation,CodeBudget,IndicateurGestion,CodePosteSecondaire,MatriculeGestionnaire,HeuresSemaineContrat,HeuresJourContrat,JoursTravailléesSemaine,DateEffet
0,35342,6585,397,23604,2815,0,7507,20208,40,8,5,1964-08-15
1,45985,6585,397,26946,2815,0,7507,20208,40,8,5,1973-10-09
2,31109,6203,348,37790,2950,1,79050,NaN,40,8,5,2003-06-18


## 2. Niveau 1 – une comparaison « identique / différent » naïve produit beaucoup de bruit

In [3]:
from corroborai.appariement import apparier
naif = []
paires = [p for p in apparier(d.source, d.cible) if p.i_src is not None and p.i_cib is not None]
for ch in CHAMPS:
    diff = sum(str(d.source.loc[p.i_src, ch.sources[0]]) != str(d.cible.loc[p.i_cib, ch.cible]) for p in paires)
    naif.append((ch.cible, ch.sources[0], diff))
naif = pd.DataFrame(naif, columns=["Champ cible", "Champ source", "Lignes « différentes » (brut)"])
print("Total de différences brutes :", naif.iloc[:, 2].sum(), "sur", len(paires) * len(CHAMPS), "contrôles")
naif

Total de différences brutes : 211 sur 550 contrôles


,Champ cible,Champ source,Lignes « différentes » (brut)
0,personId,Matricule,0
1,givenName,PrénomUsuel,0
2,surname,NomFamille,0
3,contactEmail,PrénomUsuel,22
4,onboardDate,DateEmbaucheRécente,22
5,siteName,LibelléSite,2
6,siteCode,CodeSite,0
7,divisionId,CodeDirection,0
8,divisionName,CodeDirection,22
9,divisionCode,CodeImputation,0


## 3. Niveaux 1-3 – exécution du moteur hybride

In [4]:
res = corroborer(d, Parametres(date_reference=date(2026, 10, 3)))
t = res.tableau()
print(res.stats())
t.pivot_table(index="Champ cible", columns="Verdict", values="ID", aggfunc="count", fill_value=0)

{'total': 551, 'Conforme': 383, 'Écart justifié': 152, 'Anomalie': 16, 'ambigus_ia': 55, 'origines': {'Déterministe – comparaison directe': 341, 'IA – arbitre local (hors-ligne)': 55, 'Déterministe – normalisation': 22, 'Déterministe – règle métier': 133}, 'affectations_source': 23, 'affectations_cible': 22, 'employes_anomalie': 11}


Verdict,Anomalie,Conforme,Écart justifié
Champ cible,,,
(affectation entière),1,0,0
assignmentEndDate,0,22,0
assignmentStartDate,3,19,0
contactEmail,0,0,22
contractTypeCode,4,0,18
dailyHoursOverride,3,18,1
detailedStatus,0,22,0
divisionCode,0,22,0
divisionId,0,22,0


### Qui décide ?
Chaque verdict porte son **origine** : comparaison directe, normalisation, règle métier, IA (LLM ou arbitre local) ou expert.

In [5]:
t["Origine de la décision"].value_counts()

Origine de la décision
Déterministe – comparaison directe    341
Déterministe – règle métier           133
IA – arbitre local (hors-ligne)        55
Déterministe – normalisation           22
Name: count, dtype: int64

## 4. Hypothèse H1 – la règle « date d'effet du poste » est ambiguë
Le mapping dit « date la **plus ancienne** » entre la date du changement d'unité administrative et la date d'effet du poste.
On teste empiriquement les deux lectures contre la cible :

In [6]:
pd.DataFrame(res.validation_hypotheses)

,Hypothèse,Interprétation,Concordances,Retenue
0,H1 – R-DATE-DEBUT,"max(date unité adm., date entrée poste)",19/22,True
1,H1 – R-DATE-DEBUT,"min(date unité adm., date entrée poste) – lecture littérale",0/22,False


La lecture littérale (min) ne concorde avec **aucune** ligne ; la lecture « plus récente » (max) concorde avec 19/22 lignes,
et est cohérente avec la règle symétrique de fin (« plus ancienne » des dates de fin). Une affectation ne peut débuter avant
que l'unité courante soit applicable. Les 3 lignes restantes sont précisément des anomalies (voir § 6). Le mode est paramétrable.

## 5. Niveau 3 – l'IA sur les cas ambigus
Les écarts non résolus par les règles reçoivent des **signaux** calculés sur tout le jeu de données (transcodage systémique,
préfixe d'environnement, valeur par défaut de la cible, valeur issue du détail du poste, permutation, incohérence interne…).
S'il existe au moins une preuve *explicative*, le cas est **ambigu** et part à l'arbitrage IA (LLM + arbitre local).

In [7]:
ia = t[t["Signature"] != ""]
ia.groupby(["Signature", "Verdict"]).agg(cas=("ID", "count"), confiance=("Confiance", "mean"))

,,cas,confiance
Signature,Verdict,,
assignmentStartDate|S-DETAIL-SANS-CHANGEMENT-UNITE+S-VALEUR-DETAIL-POSTE,Anomalie,3,0.85
contactEmail|S-PREFIXE-ENV+S-STRUCTURE-CONFORME+S-TRANSCODAGE,Écart justifié,22,0.80
dailyHoursOverride|S-DEFAUT-CIBLE+S-QUART-COHERENT+S-SOURCE-VIDE+S-VALEUR-DETAIL-POSTE,Écart justifié,1,0.70
dailyHoursOverride|S-DEFAUT-CIBLE+S-VALEUR-DETAIL-POSTE+S-VALEUR-SOURCE-EXPLICITE,Anomalie,3,0.85
positionName|S-INCOHERENCE-INTERNE+S-STRUCTURE-CONFORME+S-TRANSCODAGE,Écart justifié,22,0.75
weeklyHoursOverride|S-DEFAUT-CIBLE+S-QUART-COHERENT+S-SOURCE-VIDE+S-VALEUR-DETAIL-POSTE,Écart justifié,1,0.70
weeklyHoursOverride|S-DEFAUT-CIBLE+S-VALEUR-DETAIL-POSTE+S-VALEUR-SOURCE-EXPLICITE,Anomalie,3,0.85


### Exemple : `positionName` – un transcodage 1-pour-1 sur 100 % des lignes

In [8]:
pn = t[t["Champ cible"] == "positionName"][["Valeur attendue", "Valeur cible"]].drop_duplicates().sort_values("Valeur attendue")
pn

,Valeur attendue,Valeur cible
362,6031-Empl6031,0485-Empl0485
337,6051-Empl6051,0449-Empl0449
37,6203-Empl6203,4367-Empl4367
11,6585-Empl6585,3649-Empl3649
237,6622-Empl6622,4368-Empl4368
487,6725-Empl6725,4051-Empl4051
412,6754-Empl6754,0509-Empl0509
312,6900-Empl6900,5123-Empl5123
437,6972-Empl6972,5126-Empl5126


Chaque code d'emploi source correspond à **un seul** code cible et inversement : c'est un artefact systémique
(table de transcodage / anonymisation), pas 22 erreurs indépendantes → écart justifié **à confirmer**, confiance 75 %.

### Exemple : heures – valeur explicite écrasée par une valeur par défaut

In [9]:
for id_ in t[(t["Champ cible"] == "weeklyHoursOverride") & (t["Signature"] != "")]["ID"]:
    c = next(x for x in res.constats if x.id == id_)
    print(f"{c.matricule}: source={c.valeur_source} cible={c.valeur_cible} → {c.verdict} ({c.confiance:.0%})")
    for s in c.signaux: print("   ", s.nature, s.code, "-", s.description)

2911996: source=35 cible=40 → Anomalie (85%)
    explicatif S-DEFAUT-CIBLE - La cible contient « 40 » sur 22/22 lignes alors que la source présente 3 valeurs distinctes → valeur par défaut probable.
    explicatif S-VALEUR-DETAIL-POSTE - La valeur cible 40 correspond aux heures contractuelles du poste 12548 (HeuresSemaineContrat = 40).
    aggravant S-VALEUR-SOURCE-EXPLICITE - La source porte une valeur explicite 35 avec le code de quart 8 (≠ quart majoritaire 23 : horaire spécifique de l'employé).
4402456: source=36 cible=40 → Anomalie (85%)
    explicatif S-DEFAUT-CIBLE - La cible contient « 40 » sur 22/22 lignes alors que la source présente 3 valeurs distinctes → valeur par défaut probable.
    explicatif S-VALEUR-DETAIL-POSTE - La valeur cible 40 correspond aux heures contractuelles du poste 26586 (HeuresSemaineContrat = 40).
    aggravant S-VALEUR-SOURCE-EXPLICITE - La source porte une valeur explicite 36 avec le code de quart 9 (≠ quart majoritaire 23 : horaire spécifique de l'em

## 6. Anomalies à investiguer (triées par priorité)

In [10]:
t[t["Verdict"] == "Anomalie"].sort_values("Priorité", ascending=False)[
    ["ID", "Niveau priorité", "Matricule", "Champ cible", "Valeur attendue", "Valeur cible", "Confiance", "Origine de la décision"]]

,ID,Niveau priorité,Matricule,Champ cible,Valeur attendue,Valeur cible,Confiance,Origine de la décision
25,C-0026,Haute,1545850,(affectation entière),Affectation présente,(absente),0.97,Déterministe – règle métier
142,C-0143,Haute,2762457,contractTypeCode,JWN,WHX,0.99,Déterministe – règle métier
167,C-0168,Haute,4625374,contractTypeCode,WHX,JWN,0.99,Déterministe – règle métier
417,C-0418,Haute,7254364,contractTypeCode,XFLR,JWN,0.99,Déterministe – règle métier
392,C-0393,Haute,3712987,contractTypeCode,JWN,XFLR,0.99,Déterministe – règle métier
348,C-0349,Haute,2911996,dailyHoursOverride,7,8,0.85,IA – arbitre local (hors-ligne)
372,C-0373,Haute,4402456,weeklyHoursOverride,36,40,0.85,IA – arbitre local (hors-ligne)
347,C-0348,Haute,2911996,weeklyHoursOverride,35,40,0.85,IA – arbitre local (hors-ligne)
497,C-0498,Haute,7683990,weeklyHoursOverride,36,40,0.85,IA – arbitre local (hors-ligne)
373,C-0374,Haute,4402456,dailyHoursOverride,7.2,8,0.85,IA – arbitre local (hors-ligne)


## 7. Les trois types de cas exigés pour la démonstration

In [11]:
def fiche(id_):
    c = next(x for x in res.constats if x.id == id_)
    print(f"[{c.verdict}] {c.id} – matricule {c.matricule} – {c.champ_cible} ({c.regle}, {c.origine}, {c.confiance:.0%})")
    print(f"  source={c.valeur_source} | attendu={c.valeur_attendue} | cible={c.valeur_cible}")
    print("  Justification :", c.justification)
    for tr in c.trace: print("   >", tr)
    print()
fiche(t[(t["Verdict"] == "Conforme") & (t["Champ cible"] == "onboardDate")]["ID"].iloc[0])
fiche(t[(t["Champ cible"] == "statusReasonCode") & (t["Valeur cible"] == "170")]["ID"].iloc[0])
fiche(t[(t["Verdict"] == "Anomalie") & (t["Champ cible"] == "assignmentStartDate")]["ID"].iloc[0])

[Conforme] C-0005 – matricule 1545850 – onboardDate (R-DIRECT, Déterministe – normalisation, 100%)
  source=1995-02-09 00:00:00 | attendu=1995-02-09 | cible=1995-02-09T00:00:00.000Z
  Justification : Valeurs identiques après normalisation : Dates ramenées au format AAAA-MM-JJ (heure, fuseau et format ISO 'Z' ignorés).
   > DateEmbaucheRécente = « 1995-02-09 » (copie directe)

[Écart justifié] C-0090 – matricule 7603160 – statusReasonCode (R-SIT-CAD, Déterministe – règle métier, 100%)
  source=CodeStatutEmploi=2 ; CodeRaisonStatut=807 ; CodeSuspensionAccès=2 | attendu=170 | cible=170
  Justification : Écart expliqué par la règle R-SIT-CAD (Code d'absence Remphor (cf_CAD)) : CodeSuspensionAccès (source) = 2 ; Jointure motif 807 → CodeGestionAccès = 2 (cohérent) ; Absence → code Remphor associé au motif 807 = 170
   > CodeSuspensionAccès (source) = 2
   > Jointure motif 807 → CodeGestionAccès = 2 (cohérent)
   > Absence → code Remphor associé au motif 807 = 170

[Anomalie] C-0046 – matric

## 8. Observations de qualité (hors verdict) et export

In [12]:
from corroborai import rapport
rapport.exporter_excel(res, "sorties/rapport_corroboration.xlsx")
pd.DataFrame(res.observations)

,Code,Gravité,Objet,Description
0,O-MOTIF-INCONNU,Faible,CodeRaisonStatut 651,2 ligne(s) source (2 employé(s)) ont un motif absent de la table « Motif de la situati...
1,O-MOTIF-INCONNU,Faible,CodeRaisonStatut 697,2 ligne(s) source (1 employé(s)) ont un motif absent de la table « Motif de la situati...
2,O-MOTIF-INCONNU,Faible,CodeRaisonStatut 703,17 ligne(s) source (15 employé(s)) ont un motif absent de la table « Motif de la situa...
3,O-RETOUR-DEPASSE,Moyenne,Matricule 7603160,Absence complète avec date de retour prévue le 2015-06-17 (dépassée de 11.3 ans). Les ...
4,O-RETOUR-DEPASSE,Moyenne,Matricule 2911996,Absence complète avec date de retour prévue le 2010-05-28 (dépassée de 16.4 ans). Les ...
5,O-SOURCE-VIDE,Moyenne,Matricule 3712987,HeuresNormeHebdo est vide dans le système A (poste 86337) : la valeur cible ne peut pa...
6,O-SOURCE-VIDE,Moyenne,Matricule 3712987,HeuresNormeQuotidienne est vide dans le système A (poste 86337) : la valeur cible ne p...
7,O-HISTORIQUE-ANTERIEUR,Info,15 postes,L'historique « détail du poste » de ces postes est entièrement antérieur à l'embauche ...
